In [2]:
import threading
import pandas as pd

STOCK_URL ='../static/data/'

class BeuData(object):
    _instance_lock = threading.Lock()

    def __new__(cls, *args, **kwargs):
        if not hasattr(cls, '_instance'):
            with BeuData._instance_lock:
                if not hasattr(cls, '_instance'):
                    BeuData._instance = super().__new__(cls)

            return BeuData._instance

        
    def query_stock(self, code, start_date, end_date, frequency, ):
        # 如果code不存在列表内，直接返回
        if not self.__check_code_exist(code=code):
            print('股票代码不存在股票列表内')
            return
        else:
            # 获取股票原始数据
            raw = self.__get_data(code=code,frequency=frequency)
            # 获取复权因子数据
            adjust_factor = self.__get_adjust_factor(code=code)
            # 复权计算
            fore_adjust = self.__adjust_cal(raw=raw, adjust_factor = adjust_factor)
#             back_adjust = self.__adjust_cal(raw=raw, adjust_factor = adjust_factor)
            # 根据start_date与end_date返回数据
    

    def __check_code_exist(self, code):
        # 查询all_stock.csv中是否存在与code匹配
        try:
            code_df = pd.read_csv(STOCK_URL + 'all_stock.csv', usecols=['code'])
            count = code_df.count().code
            if count == 0:
                print('指数代码为空，请检查代码')
                return False
            else:
                is_code_in_list = code in code_df['code'].values
                return is_code_in_list
        except Exception as e:
            raise e
            
            
    def __get_data(self, code, frequency):
        try:
            if frequency == 'd':
                url = 'day/'
            elif frequency == '5':
                if self.__is_code_in_min_ignore(code=code):
                    print(f'{code} 没有分钟交易数据')
                    return
                url ='min/'
            else:
                print('Frequency should be d or 5.')
                
            df = pd.read_csv(STOCK_URL + url + code +'.csv')
            return df
        except Exception as e:
            raise e
            
            
    def __is_code_in_min_ignore(self, code):
        try:
            ignore_df = pd.read_csv(STOCK_URL + 'min_ignore.csv', usecols=['code'])
            is_code_in_min_ignore = code in ignore_df['code'].values
            return is_code_in_min_ignore
        except Exception as e:
            print(e)
            
    def __get_adjust_factor(self, code):
        try:
            adjust_factor = pd.read_csv(STOCK_URL + 'adjust_factor_data.csv')
            condition = adjust_factor['code'] == code
            filter_adjust = adjust_factor[condition]
            return filter_adjust
        except Exception as e:
            print(e)
            
    def __adjust_cal(self, raw, adjust_factor):
        print(adjust_factor)
        operate_date = adjust_factor['dividOperateDate']
        print(operate_date)
        count = operate_date.count()
        for i in range(count):
            print (adjust_factor.iloc[i].foreAdjustFactor)


beu_data = BeuData()


beu_data.query_stock(code='sh.600121',start_date='1999-01-01', end_date='2014-01-01', frequency='5')
# beu_data.check_code_exist(code='sh.600011')

code dividOperateDate  foreAdjustFactor  backAdjustFactor  \
1550  sh.600121       2000-05-23          0.759727          2.722876   
1551  sh.600121       2001-06-01          0.760661          2.726221   
1552  sh.600121       2005-06-15          0.780945          2.798921   
1553  sh.600121       2007-06-20          0.787469          2.822303   
1554  sh.600121       2009-06-05          0.796347          2.854122   
1555  sh.600121       2012-04-26          0.804686          2.884008   
1556  sh.600121       2013-05-14          0.820755          2.941600   
1557  sh.600121       2019-07-22          1.000000          3.584018   

      adjustFactor  
1550      2.722876  
1551      2.726221  
1552      2.798921  
1553      2.822303  
1554      2.854122  
1555      2.884008  
1556      2.941600  
1557      3.584018  
1550    2000-05-23
1551    2001-06-01
1552    2005-06-15
1553    2007-06-20
1554    2009-06-05
1555    2012-04-26
1556    2013-05-14
1557    2019-07-22
Name: dividOperateDat